# Scenario 3: Multi-Agent Research System

## 1. In One Minute

**What it is.** A coordinator agent splits a research question across subagents (web search, document analysis, synthesis, report generation). The result is a cited report.

**Domains hit:** D1 Agentic Architecture, D2 Tool Design and MCP, D5 Context Management.

**Task statements:** TS 1.2 (coordinator and subagents), 1.3 (spawning and context passing), 1.6 (task decomposition), 2.2 (structured errors), 2.3 (scoped tools), 5.3 (error propagation), 5.6 (provenance).

**Top 3 traps**
1. Blaming a subagent when the coordinator split the topic too narrowly.
2. Assuming subagents inherit the coordinator's memory. They do not; pass context in the prompt.
3. Hiding failures: returning empty results for a timeout, or averaging conflicting numbers.

## 2. Architecture Diagram

```mermaid
---
config:
  htmlLabels: false
---
flowchart TD
    User["Research question"] --> Coord["Coordinator<br>(claude-sonnet-4-5)"]
    Coord -->|"Parallel Task calls<br>(one response)"| Web["Web search subagent"]
    Coord --> Doc["Document analysis subagent"]
    Web -->|"Claims + source + date"| Synth["Synthesis subagent"]
    Doc -->|"Claims + source + date"| Synth
    Web -.->|"Structured error<br>(type, query, partial, alternatives)"| Coord
    Synth -->|"Simple fact check"| Verify["verify_fact tool<br>(scoped)"]
    Synth -->|"Complex check"| Coord
    Synth --> Report["Report generation subagent"]
    Coord --> Cover{"Coverage check:<br>every topic covered?"}
    Cover -->|"Gap"| Coord
    Report --> Out["Cited report"]
```

## 3. How a Good Answer Thinks

1. **Find where the fault is.** Good subagent output, narrow final report: the coordinator's decomposition is the cause.
2. **Hub-and-spoke.** All communication goes through the coordinator. Subagents have isolated context.
3. **Pass context explicitly.** Include findings, sources, and constraints in each subagent prompt.
4. **Parallel when independent.** Emit multiple Task calls in one coordinator response.
5. **Errors carry context.** Failure type, attempted query, partial results, alternatives. Not "search unavailable".
6. **Empty is not failure.** A working search with zero hits is a valid result. A timeout is a failure.
7. **Provenance.** Keep claim, source, and date together. Keep conflicting values with attribution; never average.
8. **Least privilege.** Give synthesis a scoped `verify_fact`, not all web tools.

### Model and Sampling Notes
- Use `claude-sonnet-4-5` for coordinators and reasoning, `claude-haiku-4-5` for workers and extraction.
- With extended thinking on: omit `temperature`, leave `top_k` and `top_p` unset, and use only `tool_choice` `auto` or `none` (forced or `any` returns HTTP 400). Forced-tool extraction calls must not use thinking.
- In tool loops, return thinking blocks with their signature unchanged.
- Loops end on `stop_reason` only (`end_turn`, `tool_use`, `max_tokens`, `stop_sequence`, `pause_turn`, `refusal`). A max-iterations counter is a safety cap, not the stop rule.

## 4. Task Statement Map

| Task statement | How it shows up here | Trap |
|---|---|---|
| TS 1.2 Coordinator | Coordinator plans, delegates, merges | Subagents talking to each other directly |
| TS 1.3 Spawning | Parallel Task calls; context passed in the prompt | Assuming shared memory |
| TS 1.6 Decomposition | Check every required topic has a subtask | Narrow split that misses topics |
| TS 2.2 Errors | Structured error returned to coordinator | Generic error or empty success |
| TS 2.3 Scoped tools | `verify_fact` for simple checks | Giving synthesis every search tool |
| TS 5.3 Propagation | Report says which sources failed (partial coverage) | Silent gaps |
| TS 5.6 Provenance | Claim-source mapping; conflicts kept | Averaging or dropping a source |

## 5. Worked Example

**Question:** "Impact of AI on creative industries." The report only covers visual arts.

1. Check subagent logs: they did their assigned work well.
2. Check the coordinator's task list: visual arts only. Music, film, writing are missing.
3. Root cause: decomposition too narrow. Fix the coordinator, for example by adding a coverage check against the required topics.
4. Re-run with subtasks for each area, in parallel. The web search for music times out. It returns `failure_type: timeout`, the attempted query, and alternatives. The coordinator retries with a narrower query.
5. Two sources give different revenue figures. The report lists both, each with source and date.
6. The report notes any area still missing, so readers see partial coverage.

Try it offline: `scenarios/scenario_3_multi_agent_research/simulation.py` (`check_decomposition_coverage`, `merge_claims`, `structured_error`). The simulation uses SEC, patent, and market subagents as examples of the same pattern.

## 7. Self-Check Checklist

- [ ] I check decomposition before blaming subagents.
- [ ] I know subagents do not inherit context.
- [ ] I can list the fields of a useful structured error.
- [ ] I separate "empty result" from "failure".
- [ ] I keep conflicting claims with sources, never averaging.

**Memory hook:** "Split wide, pass context, fail loudly, cite everything."


In [ ]:
# Step 1: Environment Setup & Simulation Import
import sys
from pathlib import Path

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scenarios.scenario_3_multi_agent_research.simulation import *

print(' Scenario Workbench Initialized: Scenario 3: Multi-Agent Research & Synthesis System')
print(' Zero live API calls, 100% offline determinism')


### 🧪 Step 2: Core Execution Flow & Happy Path Simulation
Demonstrating end-to-end pipeline execution with audit logging.


In [ ]:
# Test Consensus & Conflict Preservation
coord = ResearchCoordinator()

report = coord.run_parallel_investigation(
    "Acme Corp Earnings",
    {"sec_agent": "success", "market_agent": "success", "patent_agent": "empty"}
)
print("Coverage Status:", report.coverage_status)
print("Conflicts Preserved:", report.conflicts_preserved)
assert report.coverage_status == "COMPLETE"
assert len(report.conflicts_preserved) == 1
assert "$1.2B" in report.conflicts_preserved[0]["guidance"]
assert "$1.4B" in report.conflicts_preserved[0]["guidance"]



### ⚠️ Step 3: Edge Case Testing & Guardrail Verifications
Testing boundary conditions, safety gates, error recovery, and escalation protocols.


In [ ]:
# Test Upstream Timeout Error Handling
coord_err = ResearchCoordinator()
report_err = coord_err.run_parallel_investigation(
    "Quantum Security",
    {"sec_agent": "success", "market_agent": "success", "patent_agent": "timeout"}
)
print("Partial Coverage Status:", report_err.coverage_status)
print("Source Omissions:", report_err.source_omissions)
assert report_err.coverage_status == "PARTIAL_COVERAGE"
assert report_err.confidence_score < 1.0
assert any("patent_agent" in o for o in report_err.source_omissions)



### ✅ Step 4: Verification Summary & SLA Compliance
All assertions passed with zero unauthorized actions and 100% deterministic compliance.


### Step 5: New behaviour - coverage check, structured error, claim merge
Offline checks for the extended simulation.


In [ ]:
check = ResearchCoordinator.check_decomposition_coverage(
    ["visual arts", "music"], ["survey visual arts trends"]
)
assert check["missing_topics"] == ["music"]
failed = SpecializedSubagent("patent_agent", "PATENTS").execute_isolated_task("q", "timeout")
assert failed.structured_error["is_retryable"] is True
claims = [
    {"claim": "revenue", "value": "$1.2B", "source_name": "10-Q",
     "source_url": "u1", "date": "2024-10-01"},
    {"claim": "revenue", "value": "$1.4B", "source_name": "Release",
     "source_url": "u2", "date": "2024-10-15"},
]
assert merge_claims(claims)[0]["status"] == "CONFLICT"
print("Coverage gap:", check["missing_topics"])
